# Tarea 1 Parte B
Integrantes:

In [0]:
import pandas as pd
import re
from pyspark.sql import functions as F
from pyspark.sql.functions import expr, size, col
from pyspark.sql.functions import to_timestamp, hour
from pyspark.sql.functions import schema_of_json

In [0]:
%sql
create catalog if not exists dev;

In [0]:

%sql
create database if not exists dev.ciencias_data

In [0]:
%sql
create volume if not exists dev.ciencias_data.session_data;

#### 1. Creamos la tabla bronce en formato delta y particionado por hora para session part1.csv.

In [0]:
# Leemos el csv y cargamos
df=spark.read.format("csv").option("sep","|").option("header","true").load("/Volumes/dev/ciencias_data/session_data/sessions_part1.csv")

df=df.withColumn("_load_timestamp",F.lit(F.current_timestamp())).withColumn("_source",F.lit("Arkime"))
df = df.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

df.write \
  .format("delta") \
  .mode("overwrite") \
  .partitionBy("hour_partition") \
  .saveAsTable("dev.ciencias_data.bronze_sessions")

#### 2. Aplicamos las transformaciones al dataset

In [0]:
# Definición de funciones auxiliares
def to_snake_case(name):
    """Convierte cadenas de formato camelCase a snake_case"""
    s1 = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    return re.sub('([a-z0-9])([A-Z])', r'\1_\2', s1).lower()

# Lectura de los datos desde la capa Bronce
df_bronze = spark.table("dev.ciencias_data.bronze_sessions")

# Extracción y parseo del JSON obteniendo el esquema desde el primer registro
sample_json = df_bronze.select("data").limit(1).collect()[0][0]
schema_str = spark.range(1).select(
    schema_of_json(F.lit(sample_json))
).collect()[0][0]

# Desempaquetado del json en columnas estructuradas
df_parsed = df_bronze.withColumn(
    "json_data",
    F.from_json(F.col("data"), schema_str)
).select("json_data.*", "event_timestamp", "_load_timestamp", "_source","hour_partition")

# Cálculo de estadísticas sobre packetLen y eliminación de columnas ignoradas
df_silver = df_parsed.withColumn(
    "packet_len_sum", expr("aggregate(packetLen, cast(0 as bigint), (acc, x) -> acc + cast(x as bigint))")
).withColumn(
    "packet_len_avg", col("packet_len_sum") / size(col("packetLen"))
).withColumn(
    "packet_len_min", expr("array_min(packetLen)")
).withColumn(
    "packet_len_max", expr("array_max(packetLen)")
).drop("packetLen", "packetPos", "cert")

# Casteo de milisegundos a formato timestamp para las fechas clave
df_silver = df_silver.withColumn(
    "firstPacket", (col("firstPacket") / 1000).cast("timestamp")
).withColumn(
    "lastPacket", (col("lastPacket") / 1000).cast("timestamp")
).withColumn(
    "timestamp", (col("timestamp") / 1000).cast("timestamp")
)

# Estandarización de nombres de columnas al formato snake_case
for column in df_silver.columns:
    df_silver = df_silver.withColumnRenamed(column, to_snake_case(column))

# Escritura final en la capa Silver
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("dev.ciencias_data.silver_sessions")

# Verificación visual de los datos procesados

df_silver.display()

#### 3. Hacemos una carga incremental de session part2.csv, a su vez aplicando las mismas transformaciones que el dataset anterior

In [0]:
# Lectura del nuevo archivo y creación de columnas de partición y metadatos
df2 = spark.read.format("csv").option("sep", "|").option("header", "true").load("/Volumes/dev/ciencias_data/session_data/sessions_part2.csv")

df2 = df2.withColumn("_load_timestamp", F.lit(F.current_timestamp())).withColumn("_source", F.lit("Arkime"))
df2 = df2.withColumn(
    "event_timestamp",
    to_timestamp("timestamp")
).withColumn(
    "hour_partition",
    hour("event_timestamp")
)

# Desempaquetado del json utilizando el esquema previamente definido
df_parsed_inc = df2.withColumn(
    "json_data",
    F.from_json(F.col("data"), schema_str)
).select("json_data.*", "event_timestamp", "_load_timestamp", "_source", "hour_partition")

# Cálculo de estadísticas sobre packetLen y eliminación de columnas no requeridas
df_silver_inc = df_parsed_inc.withColumn(
    "packet_len_sum", expr("aggregate(packetLen, cast(0 as bigint), (acc, x) -> acc + cast(x as bigint))")
).withColumn(
    "packet_len_avg", col("packet_len_sum") / size(col("packetLen"))
).withColumn(
    "packet_len_min", expr("array_min(packetLen)")
).withColumn(
    "packet_len_max", expr("array_max(packetLen)")
).drop("packetLen", "packetPos", "cert")

# Transformación de marcas de tiempo en milisegundos a formato timestamp
df_silver_inc = df_silver_inc.withColumn(
    "firstPacket", (col("firstPacket") / 1000).cast("timestamp")
).withColumn(
    "lastPacket", (col("lastPacket") / 1000).cast("timestamp")
).withColumn(
    "timestamp", (col("timestamp") / 1000).cast("timestamp")
)

# Estandarización de las columnas al formato snake_case
for column in df_silver_inc.columns:
    df_silver_inc = df_silver_inc.withColumnRenamed(column, to_snake_case(column))

# Escritura incremental en la tabla Silver anexando los datos nuevos
df_silver_inc.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("dev.ciencias_data.silver_sessions")

# Verificación visual
df_silver_inc.display()

# Parte 4

In [0]:
df = spark.table("dev.ciencias_data.silver_sessions")
df.display()

## Número de sesiones por país de origen

In [0]:
df.groupBy("src_geo") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Número de sesiones por país destino

In [0]:
df.groupBy("dst_geo") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Número de sesiones por src_ip y dst_ip

In [0]:
df.groupBy("src_ip", "dst_ip") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## tot_bytes, tot_data_bytes y tot_packets por src_ip y protocol

In [0]:
df.groupBy("src_ip", "protocol") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

## tot_bytes, tot_data_bytes y tot_packets por src_mac y dst_mac

In [0]:
#Por src_mac
df_src_mac = df.withColumn("src_mac_exploded", F.explode("src_mac"))

df_src_mac.groupBy("src_mac_exploded") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

In [0]:
df_dst_mac = df.withColumn("dst_mac_exploded", F.explode("dst_mac"))

df_dst_mac.groupBy("dst_mac_exploded") \
    .agg(
        F.sum("tot_bytes").alias("sum_tot_bytes"),
        F.sum("tot_data_bytes").alias("sum_tot_data_bytes"),
        F.sum("tot_packets").alias("sum_tot_packets")
    ) \
    .orderBy(F.desc("sum_tot_bytes")) \
    .show()

## Mínimo, máximo y promedio por src_ip, srcIp, dstIp, srcMac y dstMac

In [0]:
df.groupBy("src_ip") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
df.groupBy("dst_ip") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
df_src_mac.groupBy("src_mac_exploded") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

In [0]:
df_dst_mac.groupBy("dst_mac_exploded") \
    .agg(
        F.min("tot_bytes").alias("min_tot_bytes"),
        F.max("tot_bytes").alias("max_tot_bytes"),
        F.avg("tot_bytes").alias("avg_tot_bytes"),
        F.min("tot_data_bytes"),
        F.max("tot_data_bytes"),
        F.avg("tot_data_bytes"),
        F.min("tot_packets"),
        F.max("tot_packets"),
        F.avg("tot_packets")
    ).show()

## Top 5 src_ip y src_mac con más sesiones

In [0]:
df.groupBy("src_ip") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(5) \
    .show()

In [0]:
df_src_mac.groupBy("src_mac_exploded") \
    .count() \
    .orderBy(F.desc("count")) \
    .limit(5) \
    .show()

## Número de src_mac y dst_mac por sesión

In [0]:
df.select(
    "src_ip",
    F.size("src_mac").alias("num_src_mac"),
    F.size("dst_mac").alias("num_dst_mac")
).show()

## Protocolos más usados 
Usaremos la columna Protocol

In [0]:
df.groupBy("protocol") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

## Páginas web más visitadas

In [0]:
df.groupBy("dst_asn") \
    .count() \
    .orderBy(F.desc("count")) \
    .show()

In [0]:
df.columns

df_silver.columns